In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys

In [3]:
#Entry
tperiod = 4 #time period is 4 hours long

In [5]:
# Location data
df_V = pd.read_excel('fullDist Set V Data Location.xlsx')
df_V.head

<bound method NDFrame.head of     Index      County       Type  Cluster  \
0       0         NaN  Warehouse      NaN   
1       1         NaN  Warehouse      4.0   
2       2         NaN   Supplier      NaN   
3       3         NaN   Supplier      NaN   
4       4         NaN   Supplier      NaN   
5       5         NaN   Supplier      NaN   
6       6  Georgetown    Shelter      9.0   
7       7  Georgetown    Shelter      1.0   
8       8       Horry    Shelter     18.0   
9       9       Horry    Shelter      2.0   
10     10       Horry    Shelter     16.0   
11     11       Horry    Shelter      8.0   
12     12    Berkeley    Shelter     12.0   
13     13    Berkeley    Shelter     17.0   
14     14    Berkeley    Shelter     14.0   
15     15    Berkeley    Shelter     11.0   
16     16    Berkeley    Shelter     19.0   
17     17  Charleston    Shelter      0.0   
18     18  Charleston    Shelter     15.0   
19     19  Dorchester    Shelter      7.0   
20     20  Dorchester    

In [7]:
# County data
df_C = pd.read_excel('fullDist Set C Data.xlsx')
df_C.head

<bound method NDFrame.head of        County Category of Hurricane  Shelter Total Demand  PoD Total Demand  \
0    Berkeley                 Cat 2                  6400             11596   
1  Charleston                 Cat 2                  5103              9246   
2    Colleton                 Cat 1                  1650              2500   
3  Dorchester                 Cat 1                  4460              8080   
4  Georgetown                 Cat 3                  2970              5368   
5       Horry                 Cat 3                 20234             36574   
6      Jasper                   NaN                     0                 0   

   % Evac to PubShelter  Unnamed: 5  Unnamed: 6  Unnamed: 7 All cots:  \
0                 0.190         NaN         NaN         NaN    Water    
1                 0.190         NaN         NaN         NaN     Meals   
2                 0.109         NaN         NaN         NaN       Cot   
3                 0.190         NaN         N

In [9]:
# Resource Data
df_R = pd.read_excel('fullDist R.xlsx')
df_R.head

<bound method NDFrame.head of              R  Demand Units on Truck  Pallet Units on Truck  \
0  Shelter Kit                    768                     44   
1      PoD Kit                   1728                     34   
2        Water                   2912                     26   
3        Meals                   4608                     48   
4          Cot                   3120                     52   
5      Blanket                   2496                     52   

   pallet per S-Kit  pallet per P-Kit  kit pallet meeting demand  
0          0.000000          0.000000                   0.057292  
1          0.000000          0.000000                   0.019676  
2          0.008929          0.008929                   0.000000  
3          0.010417          0.010417                   0.000000  
4          0.020833          0.000000                   0.000000  
5          0.016667          0.000000                   0.000000  >

In [11]:
#Time and distance matrices 
timeMatrix = np.loadtxt("durationFULL.txt"); # load the time duration matrix (seconds)
distanceMatrix = np.loadtxt("distanceFULL.txt");

In [13]:
df_inputs = pd.read_excel('fullDistInputs.xlsx')
df_inputs.head

<bound method NDFrame.head of    Hours in 6 days  S Curve k Value  People on Truck  Hourly Salary  \
0            144.0              0.2              4.0           15.0   
1              NaN              NaN              NaN            NaN   
2              NaN              NaN              NaN            NaN   
3              NaN              NaN              NaN            NaN   
4              NaN              NaN              NaN            NaN   
5              NaN              NaN              NaN            NaN   

   Fuel Cost Per Mile  Open Cost V0  Open Cost V1  Open Cost V2  Open Cost V3  \
0                 0.1           0.0        1000.0        1250.0         500.0   
1                 NaN           NaN           NaN           NaN           NaN   
2                 NaN           NaN           NaN           NaN           NaN   
3                 NaN           NaN           NaN           NaN           NaN   
4                 NaN           NaN           NaN           NaN    

In [15]:
# Load the K_ijt data from the Excel file
df_K = pd.read_excel('K_ijt_data_fixed.xlsx')
df_K.head()

,From_Node,To_Node,Time_Period,Capacity
0,0,0,0,98
1,0,0,1,58
2,0,0,2,101
3,0,0,3,186
4,0,0,4,132


In [16]:
# Read the Excel file
file_path = 'demand_data_.xlsx'
sheet_names = ['Demand_1', 'Demand_2']  # Replace with actual sheet names if different

# Initialize dictionary to store demand data for d_its
d_its = {}

for s, sheet in enumerate(sheet_names, start=1):  # Scenario index starts from 1
    # Read the sheet into a DataFrame
    df = pd.read_excel(file_path, sheet_name=sheet, header=1, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    # Convert DataFrame to a numpy array for easier access
    demand_array = df.values

    # Iterate over all i, t to store in d_its
    for i in range(demand_array.shape[0]):  # Iterate over rows (locations)
        actual_location = i  # Adjust this mapping as necessary to match your locations
        for t in range(demand_array.shape[1]):  # Iterate over columns (time periods)
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 1:", d_its[(6, 1, 1)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(6, 1, 2)])

# Now, d_its[(i, t, s)] can be used as the parameter in constraints



Sample demand data for location 6, time 1, scenario 1: 148.014022269498
Sample demand data for location 6, time 1, scenario 2: 162.81542449644783


In [17]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)

# Populate subsets based on the type of location
for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 4  # Assuming a time period length (can be set dynamically)
T = int(df_inputs['Hours in 6 days'][0] / tperiod)  # Total number of time periods
print('Time periods:', T)

# Time Period Subsets
T_Bef = list(range(int(T / 2)))  # Before landfall
T_Aft = list(range(int(T / 2), T))  # After landfall
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Scenario Set and Scenario Probabilities
S = ['S1', 'S2']  # Define the two scenarios
P_s = { 'S1': 0.5, 'S2': 0.5 }  # Equal probability for each scenario

# Print Scenario Set and Probabilities to verify
print('Scenarios:', S)
print('Scenario Probabilities:', P_s)


All counties: 7
All locations: 43
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 21 [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
RSA locations: 9 [27, 28, 29, 30, 31, 32, 33, 34, 35]
PoD locations: 7 [36, 37, 38, 39, 40, 41, 42]
All resources: 6
Time periods: 36
Time Before Landfall: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Time After Landfall: [18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
Scenarios: ['S1', 'S2']
Scenario Probabilities: {'S1': 0.5, 'S2': 0.5}


In [18]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[23, 42]
Number of arcs: 1400
Number of arcs eliminated by this arc set: 449


In [21]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])

#starting inventory I_ir0[I][R]
Inv=[]
for i in range(V):
    Inv.extend([df_V['S-Kit StartInv'][i],
                  df_V['P-Kit StartInv'][i],
                  df_V['Water StartInv'][i],
                  df_V['Food StartInv'][i],
                  df_V['Cot StartInv'][i],
                  df_V['Blanket StartInv'][i]])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)

#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])

#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print(K_ijt)

In [25]:
# Define the model
m = gp.Model("Two-Stage Stochastic Programming Model")

# Set the optimization direction
m.modelSense = GRB.MINIMIZE

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [27]:
# Define decision variables 
# i,j is position, r is commodity, t is time, and s is scenario

# Define first-stage decision variables
# I: Inventory of commodity r at node i at the start of time period t
I = {}  # Dictionary to hold inventory variables

for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = m.addVar(vtype=GRB.CONTINUOUS, obj=0, name="I_%s_%s_%s" % (i, r, t))

# Define second-stage decision variables
U = {}  # Demand shortage at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
            U[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, obj=0, name="U_%s_%s_%s" % (i, t, s))

S = {}  # Demand satisfied at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
            S[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, obj=0, name="S_%s_%s_%s" % (i, t, s))

f = {}  # Quantity of commodity r traveling on link (i, j) at time period t in scenario s

for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only define f_ijrt^s if (i, j) is a valid link
            for r in range(R):
                for t in range(T):
                    for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
                        f[i, j, r, t, s] = m.addVar(vtype=GRB.CONTINUOUS, obj=0, name="f_%s_%s_%s_%s_%s" % (i, j, r, t, s))


In [29]:
# Define constraints

#Starting Value of I[t=0]
for i in range(V):
    for r in range(R):
        m.addConstr(I[i,r,0]==I_ir0[i][r])

# Update demand shortage for all locations, time periods (except t=0), and scenarios
for i in range(V):
    for t in range(1, T):  # Starts from t=1 as t=0 is the initial period
        for s in range(1, len(S) + 1):  # Scenario index starts from 1 (consistent with d_its definition)
            m.addConstr(U[i, t, s] == U[i, t-1, s] - S[i, t, s] + d_its[(i, t, s)], 
                        name="DemandShortageUpdate_%s_%s_%s" % (i, t, s))

# Initial demand shortage for all locations and scenarios at time period t=0
for i in range(V):
    for s in range(1, len(S) + 1):  # Scenario index starts from 1 (consistent with d_its definition)
        m.addConstr(U[i, 0, s] == d_its[(i, 0, s)], name="InitialDemandShortage_%s_%s" % (i, s))


# Demand satisfaction constraint for all locations, time periods, and scenarios
for i in range(V):
    for t in range(T):
        for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
            m.addConstr(S[i, t, s] <= U[i, t, s], name="DemandSatisfaction_%s_%s_%s" % (i, t, s))

# Inventory update constraint for all locations, commodities, time periods (except last), and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T - 1):  # Time periods from 0 to T-2 for inventory update at t+1
            for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
                # Define the outgoing and incoming flows
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                incoming_flow = gp.quicksum(f[j, i, r, t - t_ijt[j, i] + 1, s]
                                            for j in range(V) if (j, i) in A and t - t_ijt[j, i] + 1 >= 0)
                
                # Define the constraint
                m.addConstr(
                    I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_r[r],
                    name="InventoryUpdate_%s_%s_%s_%s" % (i, r, t, s)
                )

# Inventory constraint for all locations, commodities, time periods, and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                
                # Define the constraint
                m.addConstr(
                    I[i, r, t] >= outgoing_flow + S[i, t, s] * g_r[r],
                    name="InventorySufficiency_%s_%s_%s_%s" % (i, r, t, s)
                )

# Flow capacity constraint for all links, time periods, and scenarios
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in range(len(S)):  # Assuming S is a list of scenarios, e.g., ['S1', 'S2']
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        m.addConstr(
                            gp.quicksum(w_r[r] * f[i, j, r, t, s] for r in range(R)) <= K_ijt[i][j][t],
                            name="FlowCapacity_%s_%s_%s_%s" % (i, j, t, s)
                        )


In [31]:
# Set up the objective function for the two-stage stochastic programming model
objective = gp.quicksum(
    P_s[s] * gp.quicksum(
        p_i[i] * U[i, t, s]  # Penalty cost for demand shortage
        for i in range(V) for t in range(T)
    )
    for s in range(len(S))
)

# Add the objective function to the model
m.setObjective(objective, GRB.MINIMIZE)

In [33]:
# Optimize the model
m.optimize()

# Check if the model has an optimal solution
if m.status == GRB.OPTIMAL:
    # Print the objective value
    print(f"Optimal Objective Value: {m.ObjVal}")
    
    # Print the values of the decision variables
    #print("\nDecision Variable Values:")
    #for v in m.getVars():
     #   print(f"{v.VarName}: {v.X}")
else:
    print("The model did not solve to optimality. Status code:", m.status)


Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 258 rows, 12384 columns and 258 nonzeros
Model fingerprint: 0xc2aebb4b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+02, 6e+02]
Presolve removed 258 rows and 12384 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Optimal Objective Value: 0.0

Decision Variable Values:
I_0_0_0: 0.0
I_0_0_1: 0.0
I_0_0_2: 0.0
I_0_0_3: 0.0
I_0_0_4: 0.0
I_0_0_5: 0.0
I_0_0_6: 0.0
I_0_0_7: 0.0
I_0_0_8: 0.0
I_0_